In [16]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [17]:
import sys
sys.path.append("src")

from preprocessing import load_config
from train import load_data, prepare_xy, get_models, train_and_evaluate
from evaluate import evaluate_model, plot_roc_curves, plot_feature_importance, simulate_ab_test
from evaluate import run_full_evaluation
import numpy as np
from sklearn.model_selection import train_test_split

In [18]:
cfg = load_config("config/config.yaml")

In [19]:
print("[Version A] duration 포함")
results_with = train_and_evaluate(cfg, drop_leakage=False)

print("[Version B] duration 제외")
results_without = train_and_evaluate(cfg, drop_leakage=True)

[Version A] duration 포함

[logistic_regression_with_duration] 학습 중...
  CV AUC: 0.8532 ± 0.0016

[random_forest_with_duration] 학습 중...
  CV AUC: 0.8799 ± 0.0050

[xgboost_with_duration] 학습 중...


c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:199: UserWarning: [22:23:19] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  CV AUC: 0.8695 ± 0.0080
[Version B] duration 제외

[logistic_regression_no_duration] 학습 중...
  CV AUC: 0.6647 ± 0.0041

[random_forest_no_duration] 학습 중...
  CV AUC: 0.7014 ± 0.0086

[xgboost_no_duration] 학습 중...
  CV AUC: 0.6942 ± 0.0112


c:\Users\color\AppData\Local\Programs\Python\Python310\lib\site-packages\xgboost\training.py:199: UserWarning: [22:23:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [24]:
df = load_data(cfg)
model_names = ["logistic_regression", "random_forest", "xgboost"]

all_results = {}

for drop_leakage, suffix in [(False, "_with_duration"), (True, "_no_duration")]:
    X, y = prepare_xy(df, cfg, drop_leakage=drop_leakage)
    X = X.select_dtypes(include=[np.number])

    _, X_test, _, y_test = train_test_split(
        X, y,
        test_size=cfg["model"]["test_size"],
        random_state=cfg["model"]["random_state"],
        stratify=y
    )
    eval_results = run_full_evaluation(cfg, model_names, X_test, y_test, suffix=suffix)
    all_results[suffix] = eval_results


[logistic_regression_with_duration]
  ROC-AUC : 0.8521
  Avg Precision: 0.8349
              precision    recall  f1-score   support

           0       0.75      0.84      0.79      1175
           1       0.79      0.69      0.74      1058

    accuracy                           0.77      2233
   macro avg       0.77      0.77      0.77      2233
weighted avg       0.77      0.77      0.77      2233

[SKIP] logistic_regression_with_duration: feature_importances_ 없음

[random_forest_with_duration]
  ROC-AUC : 0.8758
  Avg Precision: 0.8555
              precision    recall  f1-score   support

           0       0.82      0.79      0.81      1175
           1       0.78      0.81      0.79      1058

    accuracy                           0.80      2233
   macro avg       0.80      0.80      0.80      2233
weighted avg       0.80      0.80      0.80      2233

[INFO] Feature importance 저장: random_forest_with_duration

[xgboost_with_duration]
  ROC-AUC : 0.8590
  Avg Precision: 0.8328


In [25]:
best = max(eval_results, key=lambda r: r["auc"])
print(f"Best model: {best['name']} | AUC: {best['auc']:.4f}")

simulate_ab_test(y_test, best["y_prob"],
                 top_k_ratio=cfg["thresholds"]["top_k_ratio"])

Best model: random_forest_no_duration | AUC: 0.6883

[A/B 테스트 시뮬레이션]
  Control  전환율: 0.474 (대상 2233명)
  Treatment 전환율: 0.703 (대상 669명, 상위 30%)
  접촉 비용 절감율: 70.0%
  전환율 개선:     48.3%
